<a href="https://colab.research.google.com/github/AgentiAI-neural-decoding/agentic-eeg-decoder/blob/main/colab_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic EEG Decoder — Colab Training

Pipeline neural decoding beban kognitif berbasis Agentic AI (DIKTI 2026).
Notebook: mount Google Drive → clone repo privat → install → jalankan eksperimen di **GPU CUDA**.

**Prasyarat (sekali):**
1. Runtime → Change runtime type → **GPU**.
2. **PAT GitHub** (scope `repo`) — dimasukkan via `getpass` (tak tersimpan).
3. Struktur Drive `MyDrive/agentic-eeg/`:
   - `features/` ← `*.npz` (~393 MB) untuk eksperimen Tahap 04/05.
   - `derivatives.zip` ← arsip epoch `.fif` (~14 GB) untuk foundation model Tahap 06.
   - `checkpoints/` ← output (otomatis).


## 1. Cek GPU


In [4]:
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Aktifkan GPU: Runtime > Change runtime type > GPU'

GPU 0: Tesla T4 (UUID: GPU-4bebabfc-a001-8f40-0b7a-1f1a0d8eb2df)
torch 2.11.0+cu128 | CUDA: True


## 2. Mount Google Drive


In [5]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/agentic-eeg'
import os; os.makedirs(f'{DRIVE}/checkpoints', exist_ok=True)
!ls -la {DRIVE}

Mounted at /content/drive
total 16278622
drwx------ 2 root root        4096 Jun  1 07:27 checkpoints
-rw------- 1 root root 16669300231 Jun  1 07:42 derivatives.zip
drwx------ 2 root root        4096 May 31 13:27 features


## 3. Clone repo privat (via PAT)


In [6]:
import os, getpass, subprocess
REPO_DIR = '/content/agentic-eeg-decoder'
ORG_REPO = 'AgentiAI-neural-decoding/agentic-eeg-decoder'
if not os.path.exists(REPO_DIR):
    pat = getpass.getpass('GitHub PAT: ')
    subprocess.run(['git','clone',f'https://{pat}@github.com/{ORG_REPO}.git',REPO_DIR], check=True)
    del pat
else:
    subprocess.run(['git','-C',REPO_DIR,'pull'], check=True)
%cd {REPO_DIR}
!git log --oneline -1

/content/agentic-eeg-decoder
9fc80b4 (HEAD -> main, origin/main, origin/HEAD) chore(06): default encoder=cbramod + device=auto + uncomment install braindecode di notebook


## 4. Install dependensi
`torch` TIDAK diinstall ulang (pakai CUDA bawaan Colab).


In [7]:
!pip install -q mne mne-bids mne-icalabel pyriemann onnxruntime huggingface_hub einops
print('OK')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.8/138.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 78.1 MB/s eta 0:00:00
OK


## 5. Sambungkan data dari Drive
- **features**: symlink (hanya 5 file, baca via mount OK).
- **derivatives**: UNZIP dari `derivatives.zip` ke disk lokal `/content` (cepat; baca 1.100 file via mount Drive lambat).


In [ ]:
import os
# features -> symlink
feat_repo = f'{REPO_DIR}/03_Features/features'
if os.path.isdir(f'{DRIVE}/features') and not os.path.exists(feat_repo):
    os.symlink(f'{DRIVE}/features', feat_repo); print('linked features')
# derivatives -> unzip ke lokal (bukan symlink)
deriv_repo = f'{REPO_DIR}/02_Preprocessing/derivatives'
if os.path.exists(f'{DRIVE}/derivatives.zip') and not os.path.isdir(deriv_repo):
    print('unzip derivatives.zip -> lokal (beberapa menit)...')
    os.system(f'unzip -q -o "{DRIVE}/derivatives.zip" -d {REPO_DIR}/02_Preprocessing/')
    print('selesai unzip')
!echo 'features:'; ls {feat_repo} 2>/dev/null | head
!echo 'derivatives .fif:'; find {deriv_repo} -name '*.fif' 2>/dev/null | wc -l

linked features
unzip derivatives.zip -> lokal (beberapa menit)...


## 5b. Verifikasi data (sebelum training)


In [ ]:
import glob, mne, warnings; warnings.filterwarnings('ignore'); mne.set_log_level('ERROR')
nf = len(glob.glob(f'{feat_repo}/*.npz'))
ncog = len(glob.glob(f'{deriv_repo}/COG-BCI/sub-*/ses-*/eeg/*.fif'))
nstew = len(glob.glob(f'{deriv_repo}/STEW/sub-*/*.fif'))
print(f'features .npz : {nf} (harap 5)')
print(f'COG-BCI .fif  : {ncog} (harap 1044)')
print(f'STEW .fif     : {nstew} (harap 86)')
# sanity 1 epoch
f0 = glob.glob(f'{deriv_repo}/COG-BCI/sub-*/ses-*/eeg/*.fif')
if f0:
    ep = mne.read_epochs(f0[0])
    print('contoh epoch:', ep.get_data().shape, '| sfreq', ep.info['sfreq'], '| ch', len(ep.ch_names))
assert nf==5 and ncog>0 and nstew>0, 'Data belum lengkap — cek langkah 5 (unzip) & upload Drive'
print('OK — data siap')

## 6. Tahap 05 — multi-task shared-backbone (CUDA)


In [ ]:
!python 05_Experiments/scripts/multitask.py --config 05_Experiments/config/experiments.json --both
!cp -v 05_Experiments/results/*.json {DRIVE}/checkpoints/ 2>/dev/null

In [ ]:
!pip install -q braindecode "pandas<3"

## 7. Tahap 06 — foundation-model frozen probing

Default encoder = **placeholder** (validasi harness). Untuk **CBraMod** (rekomendasi SOTA):
1. `!pip install braindecode` (sel di bawah).
2. Set `"encoder": "cbramod"` di `06_FoundationModel/config/fm.json` (opsi cbramod sudah diisi: braindecode/cbramod-pretrained, sfreq 200, n_times 400).
3. Bobot ter-unduh otomatis dari HuggingFace saat pertama jalan. Butuh derivatives (langkah 5) + GPU.


In [ ]:
# Untuk CBraMod, install braindecode dulu & set encoder=cbramod di config/fm.json
# !pip install -q braindecode "pandas<3"
!python 06_FoundationModel/scripts/probe_transfer.py --config 06_FoundationModel/config/fm.json --both || echo 'Pastikan derivatives ter-unzip (langkah 5); utk cbramod: pip install braindecode "pandas<3" + set encoder'
!cp -v 06_FoundationModel/results/*.json {DRIVE}/checkpoints/ 2>/dev/null

## Catatan
- Hasil JSON disalin ke `Drive/.../checkpoints/` (tahan sesi putus).
- Sesi putus → jalankan ulang dari atas (idempotent).
- CUDA membuka kernel Mamba (`mamba-ssm`) bila ingin uji backbone Mamba.
- Regenerasi fitur (bila perlu): `!python 03_Features/scripts/extract_features.py --config 03_Features/config/features.json`
